<div class="alert alert-block alert-success" style="font-family: Times New Roman">
    <h4><strong>Laboratory Task 5</strong></h4>

**1. Perform Standard Imports**

<br>

**2. Create a function called `set_seed()` that accepts `seed: int` as a parameter, this function must return nothing but just set the seed to a certain value.**

<br>

**3. Create a NumPy array called "arr" that contains 6 random integers between 0 (inclusive) and 5 (exclusive), call the `set_seed()` function and use `42` as the seed parameter.**

<br>

**4. Create a tensor "x" from the array above**

<br>

**5. Change the dtype of x from `int32` to `int64`**

<br>

**6. Reshape `x` into a 3x2 tensor** <br> There are several ways to do this.

<br>

**7. Return the right-hand column of tensor `x`**

<br>

**8. Without changing x, return a tensor of square values of `x`** <br> There are several ways to do this.

<br>

**9. Create a tensor `y` with the same number of elements as `x`, that can be matrix-multiplied with `x`** <br> Use PyTorch directly (not NumPy) to create a tensor of random integers between 0 (inclusive) and 5 (exclusive). Use 42 as seed. <br> Think about what shape it should have to permit matrix multiplication.

<br>

**10. Find the matrix product of `x` and `y`.**

</div>

### 1. Standard imports

I need `numpy` for array creation and `torch` for tensors. I'm also importing `random` since I want my `set_seed()` to fix *every* source of randomness I might use later — Python's own `random` module, NumPy, and PyTorch (including the GPU generator, in case one happens to be available). If I only seed one of these, "reproducible results" only actually applies to whichever library I remembered to seed, which defeats the point.

In [21]:
import random
import numpy as np
import torch

print("NumPy version :", np.__version__)
print("Torch version :", torch.__version__)

NumPy version : 1.26.4
Torch version : 2.5.1+cpu


### 2. `set_seed()`

My function needs to seed **every** RNG I might touch later in this notebook. It takes `seed: int` and returns nothing — its only job is the side effect of seeding the generators.

In [22]:
def set_seed(seed: int) -> None:
    '''Seeds every RNG I use in this notebook so my results are reproducible.'''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

### 3. Creating `arr`: 6 random integers in [0, 5)

I'm calling `set_seed(42)` right before generating the array so I get the same numbers every time I rerun this notebook. `np.random.randint(low, high, size)` samples integers from `low` (inclusive) up to `high` (exclusive).

In [23]:
set_seed(42)

arr = np.random.randint(0, 5, 6)
print("arr:", arr)
print("dtype:", arr.dtype)

arr: [3 4 2 4 4 1]
dtype: int32


### 4. Creating tensor `x` from `arr`

I'll use `torch.from_numpy()`, which wraps my existing NumPy array as a tensor (sharing the same underlying memory) without copying the data.

In [14]:
x = torch.from_numpy(arr)
print(x)
print(x.dtype)
print(x.type())

tensor([3, 4, 2, 4, 4, 1], dtype=torch.int32)
torch.int32
torch.IntTensor


### 5. Changing `x`'s dtype from `int32` to `int64`

From what I learned in the PyTorch basics notebook, the correct way to change dtype on an existing tensor is the `.type()` method — I shouldn't re-wrap it in `torch.tensor()`, since that raises a warning/error about improper tensor cloning.

In [15]:
print("Before:", x.type())

x = x.type(torch.int64)

print("After :", x.type())

Before: torch.IntTensor
After : torch.LongTensor


### 6. Reshaping `x` into a 3x2 tensor

Both `.view()` and `.reshape()` should work here since `x` is contiguous in memory. I'll go with `.reshape()` since it's the safer general-purpose choice — it also works on non-contiguous tensors, so I don't have to think about whether `.view()` would fail.

In [16]:
x = x.reshape(3, 2)
# I could also write: x = x.view(3, 2)
print(x)
print(x.shape)

tensor([[3, 4],
        [2, 4],
        [4, 1]])
torch.Size([3, 2])


### 7. Right-hand column of `x`

Slicing works the same way I'm used to from NumPy: `:` selects all rows, and `1` selects the second (right-hand) column.

In [17]:
right_column = x[:, 1]
print(right_column)

tensor([4, 4, 1])


### 8. Squaring `x` without modifying `x`

I need to be careful here — the instructions specifically say *without changing x*. `x ** 2` creates a **new** tensor and leaves `x` untouched, which is what I want. If I used an in-place op like `x.pow_(2)` instead, that would overwrite `x`, so I'm avoiding that.

In [18]:
x_squared = x ** 2
# other ways I could have written this:
# x_squared = torch.square(x)
# x_squared = x.pow(2)

print("x squared:\n", x_squared)
print("\nx is unchanged:\n", x)

x squared:
 tensor([[ 9, 16],
        [ 4, 16],
        [16,  1]])

x is unchanged:
 tensor([[3, 4],
        [2, 4],
        [4, 1]])


### 9. Creating `y`, so it can be matrix-multiplied with `x`

Let me think through the shape requirement first. `x` has 6 elements and shape `(3, 2)`. For `torch.matmul(x, y)` (or `x @ y`) to be valid, `y`'s first dimension has to match `x`'s last dimension — so `y` needs shape `(2, k)` for some `k`. The instructions also say `y` must have the same number of elements as `x`, i.e. 6 elements total, which forces `k = 3`. So `y` needs to be shaped `(2, 3)`.

The instructions specifically want me to use **PyTorch's own** random-integer generator here (`torch.randint`), not NumPy's — so I'll call `set_seed(42)` again and sample directly with `torch.randint(low, high, size)`.

In [19]:
set_seed(42)

y = torch.randint(0, 5, (2, 3))
print(y)
print(y.shape)
print("y has the same number of elements as x:", y.numel() == x.numel())

tensor([[2, 2, 1],
        [4, 1, 0]])
torch.Size([2, 3])
y has the same number of elements as x: True


### 10. Matrix product of `x` and `y`

`x` is `(3, 2)` and `y` is `(2, 3)`, so `x @ y` should give me a `(3, 3)` result.

In [20]:
product = x @ y
# other ways I could have written this:
# product = torch.matmul(x, y)
# product = torch.mm(x, y)

print("x:\n", x)
print("\ny:\n", y)
print("\nx @ y:\n", product)
print("\nresult shape:", product.shape)

x:
 tensor([[3, 4],
        [2, 4],
        [4, 1]])

y:
 tensor([[2, 2, 1],
        [4, 1, 0]])

x @ y:
 tensor([[22, 10,  3],
        [20,  8,  2],
        [12,  9,  4]])

result shape: torch.Size([3, 3])


### What I did, step by step

| Step | What I did |
|---|---|
| `arr` | 6 random ints in $[0,5)$, seeded with 42 |
| `x` | tensor from `arr`, cast to `int64`, reshaped to $(3,2)$ |
| Right-hand column | `x[:, 1]` |
| Squared values | `x ** 2` (new tensor, `x` untouched) |
| `y` | `torch.randint(0, 5, (2, 3))`, seeded with 42, shaped so `x @ y` is valid |
| `x @ y` | a $(3,2) \times (2,3) = (3,3)$ matrix product |

Working through this made me pay a lot more attention to shapes than I usually would — especially step 9, where I had to actually reason backwards from "matrix-multiplicable with x" to figure out what shape `y` needed to be, instead of just picking a shape and hoping it works.
